In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from dotenv import load_dotenv
from zapbench.ts_forecasting import util

load_dotenv()
INFERENCE_ROOT = os.environ["INFERENCE_ROOT"]

plt.style.use("science")

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='bottom', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_ids = ["01", "02", "03", "04", "05", "06", "07", "12", "15", "16", "17", "zapbench", "janelia_pretrain"]
model_names = ["mean", "linear", "timemix"]
steps_ahead = [1, 4, 8, 16, 32]
performance_dict = {}
for subject_id in subject_ids:
  subject_name = "subject_" + subject_id if subject_id != "zapbench" and subject_id != "janelia_pretrain" else subject_id
  performance_dict[subject_id] = {}
  for model_name in model_names:
    performance_dict[subject_id][model_name] = {}
    for step_ahead in steps_ahead:
      path_to_inference = glob.glob(f"{INFERENCE_ROOT}/n_steps_4/{model_name}/{subject_name}/**/", recursive=True)[-1]
      df = util.get_per_step_metrics_from_directory(path_to_inference, metric='MAE')
      performance_dict[subject_id][model_name][step_ahead] = df['MAE'][df['steps_ahead'] == step_ahead].mean()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2, 2), dpi=200)
for subject_id in subject_ids:
  for model_name in [model_names[2]]:
    ax.plot(performance_dict[subject_id][model_name].keys(), performance_dict[subject_id][model_name].values(), 'k.', label=model_name)
    ax.set_ylim(0, 0.05)
ax.plot(performance_dict['zapbench'][model_name].keys(), performance_dict['zapbench'][model_name].values(), 'r.', label=model_name)
ax.plot(performance_dict['janelia_pretrain'][model_name].keys(), performance_dict['janelia_pretrain'][model_name].values(), 'g.', label=model_name)
# ax.legend()
plt.show()

All models

In [ ]:
subject_ids = ["01", "15", "17", "zapbench"]
model_names = ["mean", "linear", "timemix", "tsmixer", "tide"]
steps_ahead = [1, 4, 8, 16, 32]
performance_dict = {}
for subject_id in subject_ids:
  subject_name = "subject_" + subject_id if subject_id != "zapbench" and subject_id != "janelia_pretrain" else subject_id
  performance_dict[subject_id] = {}
  for model_name in model_names:
    performance_dict[subject_id][model_name] = {}
    for step_ahead in steps_ahead:
      path_to_inference = glob.glob(f"{INFERENCE_ROOT}/n_steps_4/{model_name}/{subject_name}/**/", recursive=True)[-1]
      # print(path_to_inference)
      df = util.get_per_step_metrics_from_directory(path_to_inference, metric='MAE')
      performance_dict[subject_id][model_name][step_ahead] = df['MAE'][df['steps_ahead'] == step_ahead].mean()

In [ ]:
def plot_custom_boxplot(ax, labels, boxplot_data, title, dpi=100):
    bp = ax.boxplot(
        boxplot_data[1:],
        tick_labels=labels[1:],
        patch_artist=True,
        showfliers=False,
        widths=0.4,
    )
    plt.setp(bp['medians'], color='k')
    mean_val = np.mean(boxplot_data[0])
    ax.axhline(mean_val, color='r', linestyle='--', linewidth=2, zorder=0)

    colors = ['gray']*len(boxplot_data[1:])
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('gray')

    plt.xticks(rotation=90)
    plt.tight_layout()

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(1.2)

    ax.set_xticks([])

    ax.set_yticks([0.01, 0.02, 0.03, 0.04])
    ax.set_yticklabels(['0.01', '0.02', '0.03', '0.04'], fontsize=8)
    ax.set_xticks(range(1, len(model_names)))
    model_names_to_show = ["Linear", "Time-Mix", "TSMixer", "TiDE"]
    ax.set_xticklabels(model_names_to_show, fontsize=8, rotation=90)
    ax.set_ylim(0.01, 0.04)

    ax.tick_params(axis='y', which='minor', length=0)
    ax.tick_params(axis='x', which='minor', length=0)
    ax.tick_params(axis='y', which='both', left=True, right=False, labelleft=True, direction="out", width=1.2)
    ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True, direction="out", width=1.2)

    # Add a title to the figure
    if ax.figure is not None:
        ax.figure.suptitle("MAE across subjects, $h = 4$", fontsize=8, y=0.9)
    return fig, ax

In [ ]:
fig, axs = plt.subplots(1, len(steps_ahead), figsize=(8, 2), dpi=1000)
for j, s in enumerate(steps_ahead):
  boxplot_data = []
  for m in model_names:
    boxplot_data.append([performance_dict[subject_ids[i]][m][s] for i in range(len(subject_ids))])
  plot_custom_boxplot(axs[j], model_names, boxplot_data, f"$\\Delta t={s}$")

In [ ]:
datax

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(4, 2), dpi=500)

subject_ids = list(performance_dict.keys())
model_names = list(next(iter(performance_dict.values())).keys())
colors = ['k', 'tab:blue', 'tab:orange', 'tab:green', 'tab:red']
markers = ['o', 's', 'D', '^', 'v']

for i, model_name in enumerate(model_names):
    performances = [performance_dict[sid][model_name] for sid in subject_ids]
    ax1.plot(
        range(len(subject_ids)),
        performances,
        marker=markers[i % len(markers)],
        color=colors[i % len(colors)],
        linestyle='-',
        linewidth=1.5,
        markersize=6,
        label=model_name
    )

ax1.set_ylabel("$G_x$", fontsize=12, labelpad=-10)
ax1.yaxis.label.set_position((0.0, 0.5))

# Set x-ticks to subject ids
ax1.set_xticks(range(len(subject_ids)))
ax1.set_xticklabels(subject_ids, fontsize=12)

# Optionally, set y-limits to a reasonable range for MAE
all_performances = [
    performance_dict[sid][model_name]
    for sid in subject_ids
    for model_name in model_names
]
ax1.set_ylim(0, max(all_performances)*1.1)

# Style
for spine in ax1.spines.values():
    spine.set_linewidth(1.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(which='minor', length=0)
ax1.tick_params(axis='both', labelsize=12)
ax1.tick_params(axis='x', which='major', top=False, direction="out", width=1.2, length=4)
ax1.tick_params(axis='y', which='major', right=False, direction="out", width=1.2, length=4)

ax1.legend(title="Model", fontsize=10, title_fontsize=10, loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

Pooled error across all timesteps and conditions

In [ ]:
subject_ids = ["01", "02", "03", "04", "05", "06", "07", "12", "15", "16", "17", "zapbench", "janelia_pretrain"]
model_names = ["mean", "linear", "timemix"]
performance_dict = {}
for n in [4, 32]:
  performance_dict[n] = {}
  for model_name in model_names:
    performance_dict[n][model_name] = {}
    for subject_id in subject_ids:
      subject_name = "subject_" + subject_id if subject_id != "zapbench" and subject_id != "janelia_pretrain" else subject_id
      path_to_inference = glob.glob(f"{INFERENCE_ROOT}/n_steps_{n}/{model_name}/{subject_name}/**/", recursive=True)[-1]
      df = util.get_per_step_metrics_from_directory(path_to_inference, metric='MAE')
      avg_mae = df['MAE'][31].mean()
      performance_dict[n][model_name][subject_id] = avg_mae

In [ ]:
n = 4
plt.plot([performance_dict[n]['mean'][k] for k in performance_dict[n]['timemix'].keys()], 'x')
# plt.plot([performance_dict[n]['linear'][k] for k in performance_dict[n]['timemix'].keys()])
plt.plot([performance_dict[n]['timemix'][k] for k in performance_dict[n]['timemix'].keys()])
n = 32
plt.plot([performance_dict[n]['mean'][k] for k in performance_dict[n]['timemix'].keys()])
# plt.plot([performance_dict[n]['linear'][k] for k in performance_dict[n]['timemix'].keys()])
plt.plot([performance_dict[n]['timemix'][k] for k in performance_dict[n]['timemix'].keys()])